# 📌 Descripción de entidades y campos clave del dataset *Games Features*

El dataset contiene información detallada sobre videojuegos, incluyendo metadatos, géneros, atributos comerciales y características de gameplay.  
Se organiza en **una entidad principal (GAMES_FEATURES)** y una entidad lógica **METASTORE**, que representa dónde se almacena la tabla dentro de Unity Catalog.

---

# 🧩 Entidad 1: METASTORE  *(Entidad lógica / administrativa)*

No proviene del dataset original, pero describe la ubicación de la tabla en Databricks.

| Campo | Tipo | Nulabilidad | Descripción |
|-------|------|--------------|-------------|
| CatalogName | STRING | NOT NULL | Nombre del catálogo. |
| SchemaName | STRING | NOT NULL | Nombre del esquema. |
| TableName | STRING | NOT NULL | Nombre de la tabla almacenada. |

---

# 🧩 Entidad 2: GAMES_FEATURES *(Entidad principal)*

Cada registro representa un videojuego único.

## ✔ Llave primaria (PK)
| Campo | Tipo | Llave | Descripción |
|-------|------|--------|-------------|
| QueryName | STRING | PK | Identificador único del videojuego. |

---

# ✔ Datos generales del videojuego
| Campo | Tipo | NULL | Descripción |
|-------|------|------|-------------|
| ReleaseDate | DATE | Sí | Fecha de lanzamiento. |
| Metacritic | INT | Sí | Puntuación Metacritic. |
| RecommendationCount | INT | Sí | Número de recomendaciones de usuarios. |
| IsFree | BOOLEAN | No | Indica si el juego es gratuito. |

---

# ✔ Clasificación por géneros (multietiqueta)

Todos los siguientes campos son **BOOLEAN NOT NULL**:

- GenreIsNonGame  
- GenreIsIndie  
- GenreIsAction  
- GenreIsAdventure  
- GenreIsCasual  
- GenreIsStrategy  
- GenreIsRPG  
- GenreIsSimulation  
- GenreIsEarlyAccess  
- GenreIsFreeToPlay  
- GenreIsSports  
- GenreIsRacing  
- GenreIsMassivelyMultiplayer  

---

# ✔ Información comercial
| Campo | Tipo | NULL | Descripción |
|-------|------|------|-------------|
| PriceInitial | DOUBLE | Sí | Precio original del juego. |

---

# 🔑 Resumen de llaves y nulabilidad

| Tipo | Campos |
|------|--------|
| **PK** | QueryName |
| **NOT NULL** | QueryName, IsFree, todos los géneros |
| **NULL permitido** | ReleaseDate, Metacritic, RecommendationCount, PriceInitial |



# 📷 Diagrama del modelo de datos
![](diagrama2.png)

# 📷 Configura y evidencia la infraestructura en Databricks CE
![](cap1.png)

![](cap2.png)


In [0]:
spark.version


'4.0.0'

In [0]:
import sys
sys.version


'3.12.3 (main, Aug 14 2025, 17:47:21) [GCC 13.3.0]'

En este proyecto se utilizó el sistema de almacenamiento Unity Catalog → Volumes.  
El archivo fuente se encuentra en:

/Volumes/workspace/default/top_animes/games_features/games-features-edit.csv

---

# 📌 Obtención del dataset

Se realizó de forma manual 
/Volumes/workspace/default/top_animes/games_features/games-features-edit.csv

![](cap3.png)


## ✔ Carga en Spark, Persistencia: crear tabla Delta

In [0]:
%sql
DROP TABLE IF EXISTS games_catalog.games_db.games_features;

CREATE TABLE games_catalog.games_db.games_features (
  ResponseName STRING NOT NULL,
  ReleaseDate STRING,  
  Metacritic INT,
  RecommendationCount INT,
  IsFree BOOLEAN NOT NULL,
  
  GenreIsNonGame BOOLEAN NOT NULL,
  GenreIsIndie BOOLEAN NOT NULL,
  GenreIsAction BOOLEAN NOT NULL,
  GenreIsAdventure BOOLEAN NOT NULL,
  GenreIsCasual BOOLEAN NOT NULL,
  GenreIsStrategy BOOLEAN NOT NULL,
  GenreIsRPG BOOLEAN NOT NULL,
  GenreIsSimulation BOOLEAN NOT NULL,
  GenreIsEarlyAccess BOOLEAN NOT NULL,
  GenreIsFreeToPlay BOOLEAN NOT NULL,
  GenreIsSports BOOLEAN NOT NULL,
  GenreIsRacing BOOLEAN NOT NULL,
  GenreIsMassivelyMultiplayer BOOLEAN NOT NULL,

  PriceInitial DOUBLE
)
USING DELTA;


In [0]:
from pyspark.sql.types import *
from pyspark.sql import functions as F

# Ruta al CSV en Volumes
path = "/Volumes/workspace/default/top_animes/games_features/games-features-edit.csv"

# Esquema con ReleaseDate como STRING (NO DATE)
schema = StructType([
    StructField("QueryName", StringType(), True),
    StructField("ReleaseDate", StringType(), True),   # <-- STRING
    StructField("Metacritic", IntegerType(), True),
    StructField("RecommendationCount", IntegerType(), True),
    StructField("IsFree", BooleanType(), True),

    StructField("GenreIsNonGame", BooleanType(), False),
    StructField("GenreIsIndie", BooleanType(), False),
    StructField("GenreIsAction", BooleanType(), False),
    StructField("GenreIsAdventure", BooleanType(), False),
    StructField("GenreIsCasual", BooleanType(), False),
    StructField("GenreIsStrategy", BooleanType(), False),
    StructField("GenreIsRPG", BooleanType(), False),
    StructField("GenreIsSimulation", BooleanType(), False),
    StructField("GenreIsEarlyAccess", BooleanType(), False),
    StructField("GenreIsFreeToPlay", BooleanType(), False),
    StructField("GenreIsSports", BooleanType(), False),
    StructField("GenreIsRacing", BooleanType(), False),
    StructField("GenreIsMassivelyMultiplayer", BooleanType(), False),

    StructField("PriceInitial", DoubleType(), True)
])

# Leer archivo sin convertir fecha
df_raw = (
    spark.read.format("csv")
    .option("header", "true")
    .schema(schema)
    .load(path)
)

df_raw.show(20, truncate=False)
df_raw.printSchema()

# Guardar en Delta
tabla_final = "games_catalog.games_db.games_features"

df_raw.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(tabla_final)

print("Tabla creada en:", tabla_final)


+------------------------------+-----------+----------+-------------------+------+--------------+------------+-------------+----------------+-------------+---------------+----------+-----------------+------------------+-----------------+-------------+-------------+---------------------------+------------+
|QueryName                     |ReleaseDate|Metacritic|RecommendationCount|IsFree|GenreIsNonGame|GenreIsIndie|GenreIsAction|GenreIsAdventure|GenreIsCasual|GenreIsStrategy|GenreIsRPG|GenreIsSimulation|GenreIsEarlyAccess|GenreIsFreeToPlay|GenreIsSports|GenreIsRacing|GenreIsMassivelyMultiplayer|PriceInitial|
+------------------------------+-----------+----------+-------------------+------+--------------+------------+-------------+----------------+-------------+---------------+----------+-----------------+------------------+-----------------+-------------+-------------+---------------------------+------------+
|Counter-Strike                |Nov 1 2000 |88        |68991              |fals

## ✔ SECCIÓN COMPLETA: Validaciones en Spark y SQL

In [0]:
%sql
USE games_catalog.games_db;

DESCRIBE TABLE games_features;


col_name,data_type,comment
QueryName,string,null
ReleaseDate,string,null
Metacritic,int,null
RecommendationCount,int,null
IsFree,boolean,null
GenreIsNonGame,boolean,null
GenreIsIndie,boolean,null
GenreIsAction,boolean,null
GenreIsAdventure,boolean,null
GenreIsCasual,boolean,null


In [0]:
%sql
SELECT COUNT(*) AS total_registros
FROM games_features;


total_registros
12624


In [0]:
%sql
SELECT * 
FROM games_features 
LIMIT 20;


QueryName,ReleaseDate,Metacritic,RecommendationCount,IsFree,GenreIsNonGame,GenreIsIndie,GenreIsAction,GenreIsAdventure,GenreIsCasual,GenreIsStrategy,GenreIsRPG,GenreIsSimulation,GenreIsEarlyAccess,GenreIsFreeToPlay,GenreIsSports,GenreIsRacing,GenreIsMassivelyMultiplayer,PriceInitial
Counter-Strike,Nov 1 2000,88,68991,false,false,false,true,false,false,false,false,false,false,false,false,false,false,9.99
Team Fortress Classic,Apr 1 1999,0,2439,false,false,false,true,false,false,false,false,false,false,false,false,false,false,4.99
Day of Defeat,May 1 2003,79,2319,false,false,false,true,false,false,false,false,false,false,false,false,false,false,4.99
Deathmatch Classic,Jun 1 2001,0,888,false,false,false,true,false,false,false,false,false,false,false,false,false,false,4.99
Half-Life: Opposing Force,Nov 1 1999,0,2934,false,false,false,true,false,false,false,false,false,false,false,false,false,false,4.99
Ricochet,Nov 1 2000,0,1965,false,false,false,true,false,false,false,false,false,false,false,false,false,false,4.99
Half-Life,Nov 8 1998,96,12486,false,false,false,true,false,false,false,false,false,false,false,false,false,false,9.99
Counter-Strike: Condition Zero,Mar 1 2004,65,7067,false,false,false,true,false,false,false,false,false,false,false,false,false,false,9.99
Counter-Strike: Condition Zero,Mar 1 2004,65,7067,false,false,false,true,false,false,false,false,false,false,false,false,false,false,9.99
Half-Life: Blue Shift,Jun 1 2001,71,2219,false,false,false,true,false,false,false,false,false,false,false,false,false,false,4.99


In [0]:
%sql
SELECT 
  AVG(Metacritic) AS avg_metacritic,
  MIN(PriceInitial) AS min_price,
  MAX(PriceInitial) AS max_price
FROM games_catalog.games_db.games_features;


avg_metacritic,min_price,max_price
12.955640050697085,0.0,449.99


## ✔Consultas SELECT y GROUP BY

In [0]:
%sql
SELECT IsFree, COUNT(*) AS total
FROM games_catalog.games_db.games_features
GROUP BY IsFree;


IsFree,total
false,11651
true,973


In [0]:
df = spark.table("games_catalog.games_db.games_features")
df.groupBy("IsFree").count().show()



+------+-----+
|IsFree|count|
+------+-----+
| false|11651|
|  true|  973|
+------+-----+



## ✔ Conteos y muestras

✅ SQL – total de registros

In [0]:
%sql
SELECT COUNT(*) AS total_registros
FROM games_catalog.games_db.games_features;


total_registros
12624


✅ PySpark – muestra aleatoria

In [0]:
df.sample(0.05).show(10, truncate=False)


+----------------------------------------------+-----------+----------+-------------------+------+--------------+------------+-------------+----------------+-------------+---------------+----------+-----------------+------------------+-----------------+-------------+-------------+---------------------------+------------+
|QueryName                                     |ReleaseDate|Metacritic|RecommendationCount|IsFree|GenreIsNonGame|GenreIsIndie|GenreIsAction|GenreIsAdventure|GenreIsCasual|GenreIsStrategy|GenreIsRPG|GenreIsSimulation|GenreIsEarlyAccess|GenreIsFreeToPlay|GenreIsSports|GenreIsRacing|GenreIsMassivelyMultiplayer|PriceInitial|
+----------------------------------------------+-----------+----------+-------------------+------+--------------+------------+-------------+----------------+-------------+---------------+----------+-----------------+------------------+-----------------+-------------+-------------+---------------------------+------------+
|Day of Defeat                 

✅ SQL – primeros registros

In [0]:
%sql
SELECT *
FROM games_catalog.games_db.games_features
LIMIT 10;


QueryName,ReleaseDate,Metacritic,RecommendationCount,IsFree,GenreIsNonGame,GenreIsIndie,GenreIsAction,GenreIsAdventure,GenreIsCasual,GenreIsStrategy,GenreIsRPG,GenreIsSimulation,GenreIsEarlyAccess,GenreIsFreeToPlay,GenreIsSports,GenreIsRacing,GenreIsMassivelyMultiplayer,PriceInitial
Counter-Strike,Nov 1 2000,88,68991,false,false,false,true,false,false,false,false,false,false,false,false,false,false,9.99
Team Fortress Classic,Apr 1 1999,0,2439,false,false,false,true,false,false,false,false,false,false,false,false,false,false,4.99
Day of Defeat,May 1 2003,79,2319,false,false,false,true,false,false,false,false,false,false,false,false,false,false,4.99
Deathmatch Classic,Jun 1 2001,0,888,false,false,false,true,false,false,false,false,false,false,false,false,false,false,4.99
Half-Life: Opposing Force,Nov 1 1999,0,2934,false,false,false,true,false,false,false,false,false,false,false,false,false,false,4.99
Ricochet,Nov 1 2000,0,1965,false,false,false,true,false,false,false,false,false,false,false,false,false,false,4.99
Half-Life,Nov 8 1998,96,12486,false,false,false,true,false,false,false,false,false,false,false,false,false,false,9.99
Counter-Strike: Condition Zero,Mar 1 2004,65,7067,false,false,false,true,false,false,false,false,false,false,false,false,false,false,9.99
Counter-Strike: Condition Zero,Mar 1 2004,65,7067,false,false,false,true,false,false,false,false,false,false,false,false,false,false,9.99
Half-Life: Blue Shift,Jun 1 2001,71,2219,false,false,false,true,false,false,false,false,false,false,false,false,false,false,4.99


## ✔ Validación de campos específicos

In [0]:
%sql
SELECT 
  COUNT(*) FILTER (WHERE IsFree = TRUE) AS juegos_gratis,
  COUNT(*) FILTER (WHERE IsFree = FALSE) AS juegos_de_pago
FROM games_catalog.games_db.games_features;


juegos_gratis,juegos_de_pago
973,11651


In [0]:
%sql
SELECT 
  GenreIsAction,
  AVG(PriceInitial) AS avg_price
FROM games_catalog.games_db.games_features
GROUP BY GenreIsAction;


GenreIsAction,avg_price
true,8.795234419129882
false,9.649360043757591


Estas validaciones permiten confirmar que:

El esquema fue aplicado correctamente (tipos de datos, nulabilidad).

Los datos están completos y en el formato esperado.

La tabla en Delta Lake está bien creada y accesible desde SQL y PySpark.

Spark y SQL producen los mismos resultados, garantizando consistencia.

Las muestras visuales permiten inspección humana para detectar valores incorrectos.

## ✅ Ventajas y desventajas: SQL vs Spark

A lo largo del proyecto trabajé tanto con Spark (PySpark) como con SQL en Databricks. Aunque ambos sirven para manipular y analizar datos, cada uno tiene puntos fuertes distintos.
En mi caso, terminé usando Spark más que SQL, principalmente por la limpieza compleja del dataset y las transformaciones basadas en expresiones regulares.

## ✔ ¿Por qué usé más Spark (PySpark)?
Mi dataset requería varios procesos que son mucho más fáciles en PySpark que en SQL:

Limpieza de texto irregular (por ejemplo: “February 2011”, “Feb 1 2000”, rangos de fechas, valores mixtos).

Uso de expresiones regulares (regex) para normalizar campos.

Conversión de tipos (string → integer → date) de forma segura.

Manejo de errores sin que la ejecución se detenga.

Transformaciones complejas que SQL no maneja bien de forma declarativa.

Por eso, PySpark fue esencial en la etapa de preprocesamiento y limpieza, porque me permitió mayor control y flexibilidad.

Sin embargo, para consultas rápidas, validaciones, exploraciones y estructuras, SQL fue más cómodo, especialmente para:

DESCRIBE TABLE

SHOW CREATE TABLE

Validaciones con SELECT, GROUP BY

Revisar muestras (LIMIT, SHOW)

En resumen: Spark fue mejor para el trabajo pesado (ETL), y SQL fue mejor para revisar, explorar y validar.

## Comparativa: SQL vs PySpark

| Característica | SQL en Databricks | PySpark |
|----------------|-------------------|---------|
| **Facilidad de uso** | Simple y directo. Perfecto para consultas rápidas. | Requiere más código, pero es mucho más flexible. |
| **Transformaciones complejas** | Limitado para regex y lógica avanzada. | Excelente para ETL, regex y manejo de datos irregulares. |
| **Escenarios de uso en el proyecto** | Validaciones, DESCRIBE, SELECT, exploración. | Limpieza del dataset, normalización de fechas, escritura final Delta. |
| **Ventaja personal** | Comodidad y rapidez para validar y explorar. | Me permitió procesar y limpiar correctamente mi dataset. |
| **Limitaciones** | No maneja bien datos sucios o estructuras complejas. | Curva de aprendizaje un poco más alta. |


En este proyecto preferí PySpark porque me permitió limpiar datos irregulares y 
aplicar transformaciones complejas que no eran posibles con SQL de forma cómoda. 
Sin embargo, SQL fue fundamental para la parte de validación, verificación del 
esquema y consultas rápidas. Ambos se complementaron, pero Spark se ajustó mejor 
al tipo de dataset que estaba trabajando.
